# F1 Pitstop — V5 Maximum Accuracy Pipeline
**Competition:** [Playground Series S6E5](https://www.kaggle.com/competitions/playground-series-s6e5)

## Strategy
- **5 diverse base models**: CatBoost, XGBoost, LightGBM, HistGradientBoosting, ExtraTrees
- **37 engineered features**: tyre-age normalization, urgency scores, circuit pit rates, compound softness, etc.
- **LightGBM meta-learner**: replaces LogReg, with raw features injected alongside OOF predictions
- **Pseudo-labeling**: high-confidence test predictions (>0.85) added back as soft labels (weight=0.5)
- **V4 baseline OOF AUC**: 0.9476 → targeting 0.950+

In [ ]:
# Install / check deps
import subprocess, sys
for pkg in ['catboost', 'lightgbm', 'xgboost']:
    try:
        __import__(pkg)
        print(f'{pkg} ✓')
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

In [ ]:
from __future__ import annotations
import warnings, os
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier

print('numpy:', np.__version__)
print('pandas:', pd.__version__)
import catboost, lightgbm, xgboost, sklearn
print('catboost:', catboost.__version__)
print('lightgbm:', lightgbm.__version__)
print('xgboost:', xgboost.__version__)
print('sklearn:', sklearn.__version__)

In [ ]:
# ─── Config ───────────────────────────────────────────────────────────────────
TARGET     = 'PitNextLap'
ID_COL     = 'id'
FOLDS      = 5
SEED       = 42
PSEUDO_THR = 0.85   # confidence threshold for pseudo-labeling
PSEUDO_W   = 0.50   # sample weight for pseudo-labeled rows

# Data paths — Kaggle standard
DATA_DIR = '/kaggle/input/playground-series-s6e5'
TRAIN_PATH  = f'{DATA_DIR}/train.csv'
TEST_PATH   = f'{DATA_DIR}/test.csv'
SUB_PATH    = f'{DATA_DIR}/sample_submission.csv'

# If running locally, fall back to Downloads
if not os.path.exists(TRAIN_PATH):
    DATA_DIR   = os.path.expanduser('~/Downloads')
    TRAIN_PATH = f'{DATA_DIR}/train.csv'
    TEST_PATH  = f'{DATA_DIR}/test.csv'
    SUB_PATH   = f'{DATA_DIR}/sample_submission.csv'

print(f'Data from: {DATA_DIR}')

In [ ]:
# ─── Column groups ────────────────────────────────────────────────────────────
HIGH_CARD_COLS = [
    'Driver', 'Race', 'RaceYear', 'DriverCompound', 'DriverRace', 'CompoundStint',
]
SMALL_ONEHOT_COLS = ['Compound', 'Year', 'PitStop', 'Stint']

BASE_NUMERIC = [
    'LapNumber', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta',
    'Cumulative_Degradation', 'RaceProgress', 'Position_Change',
]
ENGINEERED = [
    # V4
    'EstimatedTotalLaps', 'LapsRemaining', 'TyreLifeRatio', 'TyreVsLapRatio',
    'DegradationPerLap', 'DeltaPerLap', 'PositionPct', 'StintTyreLoad',
    'LateRaceTyreLoad', 'TyreRemainingLoad', 'DeltaPositionInteraction',
    'LapTyreInteraction', 'ProgressSquared', 'TyreLifeSquared',
    # V5 new
    'TyreAgeNorm', 'UrgencyScore', 'LatePitPressure', 'DegRateAccel',
    'LapTimeAccel', 'TyreLifeNormByCompound', 'LapsRemainingNorm',
    'CompoundSoftness', 'IsSoft', 'IsHard', 'IsWet',
    'StintPositionLoad', 'TyreLifeCubed', 'CompoundPitLifeRatio',
    'CircuitPitRate', 'DriverPitFreq',
]
NUMERIC_COLS = BASE_NUMERIC + ENGINEERED

META_RAW_FEATS = [
    'TyreLife', 'RaceProgress', 'LapsRemaining', 'TyreLifeRatio',
    'UrgencyScore', 'CompoundSoftness', 'CircuitPitRate', 'DriverPitFreq',
    'TyreAgeNorm',
]
VOTE_MODELS = ['catboost', 'xgboost', 'histgb', 'lightgbm']

COMPOUND_SOFTNESS = {'SOFT': 3.0, 'MEDIUM': 2.0, 'HARD': 1.0, 'INTERMEDIATE': 0.5, 'WET': 0.0}

In [ ]:
# ─── Load data ────────────────────────────────────────────────────────────────
train_raw = pd.read_csv(TRAIN_PATH)
test_raw  = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SUB_PATH)

print(f'Train: {train_raw.shape}')
print(f'Test:  {test_raw.shape}')
print(f'Target dist: {train_raw[TARGET].value_counts().to_dict()}')
print(f'Positive rate: {train_raw[TARGET].mean():.4f}')
train_raw.head(3)

In [ ]:
# ─── Global stats (computed from full train — no leakage) ─────────────────────
def compute_global_stats(df):
    pit_rows = df[df[TARGET] == 1]
    compound_pit_life = pit_rows.groupby('Compound')['TyreLife'].median()
    circuit_pit_rate  = df.groupby('Race')[TARGET].mean()
    driver_pit_freq   = df.groupby('Driver')[TARGET].mean()
    return compound_pit_life, circuit_pit_rate, driver_pit_freq

compound_pit_life, circuit_pit_rate, driver_pit_freq = compute_global_stats(train_raw)
print('Compound typical pit life (laps):')
print(compound_pit_life.sort_values())
print('\nSample circuit pit rates:')
print(circuit_pit_rate.sort_values(ascending=False).head(8))

In [ ]:
# ─── Feature engineering ──────────────────────────────────────────────────────
def engineer_features(df, compound_pit_life, circuit_pit_rate, driver_pit_freq):
    df = df.copy()

    # Interaction keys
    df['RaceYear']      = df['Race'] + '|' + df['Year'].astype(str)
    df['DriverCompound']= df['Driver'] + '|' + df['Compound']
    df['DriverRace']    = df['Driver'] + '|' + df['Race']
    df['CompoundStint'] = df['Compound'] + '|' + df['Stint'].astype(str)

    # Race-length estimate
    df['EstimatedTotalLapsRaw'] = (df['LapNumber'] / df['RaceProgress'].replace(0, np.nan)).clip(20, 90)
    ry_totals = df.groupby('RaceYear')['EstimatedTotalLapsRaw'].transform('median')
    df['EstimatedTotalLaps'] = ry_totals.fillna(df['EstimatedTotalLapsRaw'])
    df['LapsRemaining'] = (df['EstimatedTotalLaps'] - df['LapNumber']).clip(lower=0)

    # Core ratios
    etl = df['EstimatedTotalLaps'].replace(0, np.nan)
    tl  = df['TyreLife'].replace(0, np.nan)
    ln  = df['LapNumber'].replace(0, np.nan)

    df['TyreLifeRatio']          = df['TyreLife'] / etl
    df['TyreVsLapRatio']         = df['TyreLife'] / ln
    df['DegradationPerLap']      = df['Cumulative_Degradation'] / tl
    df['DeltaPerLap']            = df['LapTime_Delta'] / tl
    df['PositionPct']            = df['Position'] / 20.0
    df['StintTyreLoad']          = df['Stint'] * df['TyreLife']
    df['LateRaceTyreLoad']       = df['TyreLife'] * df['RaceProgress']
    df['TyreRemainingLoad']      = df['TyreLife'] * df['LapsRemaining']
    df['DeltaPositionInteraction']= df['LapTime_Delta'] * df['PositionPct']
    df['LapTyreInteraction']     = df['LapNumber'] * df['TyreLife']
    df['ProgressSquared']        = df['RaceProgress'] ** 2
    df['TyreLifeSquared']        = df['TyreLife'] ** 2
    df['TyreLifeCubed']          = df['TyreLife'] ** 3
    df['LapsRemainingNorm']      = df['LapsRemaining'] / etl

    # Compound properties
    df['CompoundSoftness'] = df['Compound'].map(COMPOUND_SOFTNESS).fillna(1.0)
    df['IsSoft'] = (df['Compound'] == 'SOFT').astype(float)
    df['IsHard'] = (df['Compound'] == 'HARD').astype(float)
    df['IsWet']  = df['Compound'].isin(['INTERMEDIATE', 'WET']).astype(float)

    # Compound-adjusted tyre life (how used up vs typical pit age for this compound)
    df['CompoundPitLifeMedian'] = df['Compound'].map(compound_pit_life).fillna(20.0)
    df['TyreAgeNorm']           = df['TyreLife'] / df['CompoundPitLifeMedian'].replace(0, np.nan)
    df['CompoundPitLifeRatio']  = df['TyreAgeNorm']
    df['TyreLifeNormByCompound']= df['TyreAgeNorm']

    # Urgency / pressure signals
    df['UrgencyScore']    = (df['TyreLifeRatio'] * df['DegradationPerLap'].clip(-10, 10) * df['RaceProgress'])
    df['LatePitPressure'] = df['LapsRemainingNorm'] * df['TyreLife']
    df['StintPositionLoad'] = df['Stint'] * df['PositionPct']

    # Second-order approximations
    df['LapTimeAccel'] = df['DeltaPerLap'] * df['TyreLifeRatio']
    df['DegRateAccel'] = df['DegradationPerLap'] * df['TyreLifeRatio']

    # Circuit-level pit rate (historical)
    df['CircuitPitRate'] = df['Race'].map(circuit_pit_rate).fillna(circuit_pit_rate.mean())

    # Driver pit frequency (historical)
    df['DriverPitFreq'] = df['Driver'].map(driver_pit_freq).fillna(driver_pit_freq.mean())

    return df.replace([np.inf, -np.inf], np.nan)


fe_kwargs = dict(
    compound_pit_life=compound_pit_life,
    circuit_pit_rate=circuit_pit_rate,
    driver_pit_freq=driver_pit_freq,
)
train_df = engineer_features(train_raw, **fe_kwargs)
test_df  = engineer_features(test_raw,  **fe_kwargs)
print(f'Features after engineering: {train_df.shape[1]} columns')

In [ ]:
# ─── Fold encodings + dense matrix builder ────────────────────────────────────
def add_fold_encodings(tr, va, te):
    tr, va, te = tr.copy(), va.copy(), te.copy()
    gm = tr[TARGET].mean()
    for col in HIGH_CARD_COLS:
        stats  = tr.groupby(col)[TARGET].agg(['sum','count'])
        smooth = (stats['sum'] + 20*gm) / (stats['count'] + 20)
        freq   = tr[col].value_counts(normalize=True)
        for df in (tr, va, te):
            df[f'TE_{col}'] = df[col].map(smooth).fillna(gm)
            df[f'FE_{col}'] = df[col].map(freq).fillna(0.0)
    return tr, va, te


def to_dense(tr, va, te):
    avail = [c for c in NUMERIC_COLS if c in tr.columns]
    dense = avail + [f'TE_{c}' for c in HIGH_CARD_COLS] + [f'FE_{c}' for c in HIGH_CARD_COLS]

    def ohe(df):
        return pd.get_dummies(df[SMALL_ONEHOT_COLS].astype(str),
                              prefix=SMALL_ONEHOT_COLS, dtype=float)
    tr_s = ohe(tr); va_s = ohe(va); te_s = ohe(te)
    tr_s, va_s = tr_s.align(va_s, join='outer', axis=1, fill_value=0.0)
    tr_s, te_s = tr_s.align(te_s, join='outer', axis=1, fill_value=0.0)
    va_s, te_s = va_s.align(te_s, join='outer', axis=1, fill_value=0.0)

    tr_x = pd.concat([tr[dense].reset_index(drop=True), tr_s.reset_index(drop=True)], axis=1)
    va_x = pd.concat([va[dense].reset_index(drop=True), va_s.reset_index(drop=True)], axis=1)
    te_x = pd.concat([te[dense].reset_index(drop=True), te_s.reset_index(drop=True)], axis=1)

    # Fill NaN using train medians
    for col in tr_x.columns:
        if tr_x[col].isna().any():
            fill = tr_x[col].median()
            tr_x[col] = tr_x[col].fillna(fill)
            va_x[col] = va_x[col].fillna(fill)
            te_x[col] = te_x[col].fillna(fill)
    return tr_x, va_x, te_x


print('Encoding / matrix functions ready ✓')

In [ ]:
# ─── Model factory ────────────────────────────────────────────────────────────
def make_base(name, seed):
    if name == 'catboost':
        return CatBoostClassifier(
            iterations=700, depth=9, learning_rate=0.04,
            l2_leaf_reg=4.0, loss_function='Logloss', eval_metric='AUC',
            verbose=0, random_seed=seed,
        )
    if name == 'xgboost':
        return XGBClassifier(
            n_estimators=500, max_depth=8, learning_rate=0.04,
            subsample=0.85, colsample_bytree=0.80, min_child_weight=4,
            reg_lambda=3.0, reg_alpha=0.5, gamma=0.05,
            objective='binary:logistic', eval_metric='auc',
            tree_method='hist', random_state=seed, n_jobs=-1, verbosity=0,
        )
    if name == 'lightgbm':
        return LGBMClassifier(
            n_estimators=600, learning_rate=0.035, num_leaves=95,
            min_child_samples=25, subsample=0.85, colsample_bytree=0.80,
            reg_lambda=2.5, reg_alpha=0.3,
            random_state=seed, n_jobs=-1, objective='binary', verbose=-1,
        )
    if name == 'histgb':
        return HistGradientBoostingClassifier(
            learning_rate=0.04, max_depth=9, max_leaf_nodes=127,
            min_samples_leaf=25, l2_regularization=0.15,
            max_iter=400, random_state=seed,
        )
    if name == 'et':
        return ExtraTreesClassifier(
            n_estimators=700, min_samples_leaf=2, max_features=0.75,
            random_state=seed, n_jobs=-1,
        )
    raise ValueError(name)


def make_meta(seed):
    return LGBMClassifier(
        n_estimators=300, learning_rate=0.025, num_leaves=31,
        max_depth=4, min_child_samples=50, subsample=0.8,
        colsample_bytree=0.8, reg_lambda=3.0,
        random_state=seed, n_jobs=-1, objective='binary', verbose=-1,
    )


def predict_prob(model, X):
    return model.predict_proba(X)[:, 1]


def fit_model(name, model, X, y, w):
    if name in {'catboost','xgboost','lightgbm','histgb'}:
        model.fit(X, y, sample_weight=w)
    else:
        model.fit(X, y)


BASE_MODELS = ['catboost', 'xgboost', 'lightgbm', 'histgb', 'et']
print('Model factories ready ✓')

In [ ]:
# ─── Core stacking pass ───────────────────────────────────────────────────────
def run_stack(
    train_df, test_df,
    external_df=None, ext_weight=0.30,
    seed=42, label='P1',
):
    y = train_df[TARGET].astype(int)
    splitter = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=seed)

    oof_base  = {n: np.zeros(len(train_df)) for n in BASE_MODELS}
    test_base = {n: np.zeros(len(test_df))  for n in BASE_MODELS}
    meta_raw_oof = pd.DataFrame(np.nan,
                                index=range(len(train_df)),
                                columns=META_RAW_FEATS)

    for fold, (tr_idx, va_idx) in enumerate(splitter.split(train_df, y), 1):
        print(f'\n[{label}] ── Fold {fold}/{FOLDS} ──')

        ft = train_df.iloc[tr_idx].copy()
        fv = train_df.iloc[va_idx].copy()
        fe = test_df.copy()
        ft['_w'] = 1.0

        if external_df is not None and ext_weight > 0:
            ext = external_df.copy()
            ext['_w'] = ext_weight
            ft = pd.concat([ft, ext], ignore_index=True)

        ft, fv, fe = add_fold_encodings(ft, fv, fe)
        tr_x, va_x, te_x = to_dense(ft, fv, fe)

        tr_y = ft[TARGET].astype(int)
        va_y = fv[TARGET].astype(int)
        tr_w = ft['_w'].astype(float).values

        # Stash raw features for meta
        avail_meta = [c for c in META_RAW_FEATS if c in va_x.columns]
        meta_raw_oof.iloc[va_idx] = va_x[avail_meta].values

        for name in BASE_MODELS:
            m = make_base(name, seed + fold)
            fit_model(name, m, tr_x, tr_y, tr_w)
            vp = predict_prob(m, va_x)
            tp = predict_prob(m, te_x)
            oof_base[name][va_idx] = vp
            test_base[name] += tp / FOLDS
            print(f'  [{name}] fold-auc={roc_auc_score(va_y, vp):.5f}')

    # ── Assemble frames ───────────────────────────────────────────────────────
    oof_f  = pd.DataFrame({ID_COL: train_df[ID_COL].values, TARGET: y.values})
    test_f = pd.DataFrame({ID_COL: test_df[ID_COL].values})

    for name in BASE_MODELS:
        oof_f[name]  = oof_base[name]
        test_f[name] = test_base[name]
        print(f'[{label}] [{name}] OOF AUC = {roc_auc_score(y, oof_base[name]):.6f}')

    vote = [n for n in VOTE_MODELS if n in BASE_MODELS]
    if vote:
        oof_f['vote4']  = oof_f[vote].mean(axis=1)
        test_f['vote4'] = test_f[vote].mean(axis=1)
        print(f'[{label}] [vote4]   OOF AUC = {roc_auc_score(y, oof_f["vote4"]):.6f}')

    # ── Meta-model (LightGBM + raw features) ─────────────────────────────────
    meta_cols  = BASE_MODELS + (['vote4'] if 'vote4' in oof_f else [])
    avail_meta = [c for c in META_RAW_FEATS if c in meta_raw_oof.columns
                  and not meta_raw_oof[c].isna().all()]

    raw_means = meta_raw_oof[avail_meta].mean()
    meta_raw_oof_filled = meta_raw_oof[avail_meta].fillna(raw_means)

    oof_meta_X  = pd.concat([
        oof_f[meta_cols].reset_index(drop=True),
        meta_raw_oof_filled.reset_index(drop=True)
    ], axis=1)
    test_meta_X = test_f[meta_cols].copy()
    for col in avail_meta:
        test_meta_X[col] = raw_means[col]

    meta = make_meta(seed)
    meta.fit(oof_meta_X, y)
    oof_f['stack']  = predict_prob(meta, oof_meta_X)
    test_f['stack'] = predict_prob(meta, test_meta_X)

    auc = roc_auc_score(y, oof_f['stack'])
    print(f'\n[{label}] ★ STACK OOF AUC = {auc:.6f}')
    return oof_f, test_f, auc


print('Stack function ready ✓')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASS 1 — Full stacking run
# ══════════════════════════════════════════════════════════════════
print('=' * 60)
print('PASS 1: Base stacking')
print('=' * 60)

oof1, test1, auc1 = run_stack(train_df, test_df, seed=SEED, label='P1')
print(f'\n✅ Pass 1 complete  —  Stack OOF AUC: {auc1:.6f}')

In [ ]:
# Save Pass 1 outputs
oof1.to_csv('oof_pass1.csv', index=False)

sub1 = sample_sub.copy()
sub1[TARGET] = test1['stack'].values
sub1.to_csv('submission_pass1.csv', index=False)
print('Saved: oof_pass1.csv, submission_pass1.csv')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASS 2 — Pseudo-labeling
# ══════════════════════════════════════════════════════════════════
print('=' * 60)
print(f'PASS 2: Pseudo-labeling  (threshold={PSEUDO_THR})')
print('=' * 60)

test_preds   = test1['stack'].values
conf_mask    = (test_preds >= PSEUDO_THR) | (test_preds <= (1 - PSEUDO_THR))
n_confident  = conf_mask.sum()
print(f'High-confidence test rows: {n_confident:,} / {len(test_preds):,}  '
      f'({100*n_confident/len(test_preds):.1f}%)')

if n_confident >= 500:
    pseudo_df = test_df[conf_mask].copy()
    pseudo_df[TARGET] = test_preds[conf_mask]         # soft label (probability)
    pseudo_df['_w']   = PSEUDO_W

    print(f'Pseudo pos rate: {pseudo_df[TARGET].mean():.4f}')

    oof2, test2, auc2 = run_stack(
        train_df, test_df,
        external_df=pseudo_df,
        ext_weight=PSEUDO_W,
        seed=SEED + 100,
        label='P2',
    )

    print(f'\nPass 1 AUC: {auc1:.6f}')
    print(f'Pass 2 AUC: {auc2:.6f}  ({"BETTER ✅" if auc2 > auc1 else "no gain ⚠️"})')

    if auc2 > auc1:
        final_test = test2
        final_auc  = auc2
        oof2.to_csv('oof_pass2.csv', index=False)
        sub2 = sample_sub.copy()
        sub2[TARGET] = test2['stack'].values
        sub2.to_csv('submission_pass2.csv', index=False)
        print('Using PASS 2 as final')
    else:
        final_test = test1
        final_auc  = auc1
        print('Keeping PASS 1 as final')
else:
    print('Too few confident rows — skipping pseudo-labeling')
    final_test = test1
    final_auc  = auc1

In [ ]:
# ══════════════════════════════════════════════════════════════════
# FINAL SUBMISSION
# ══════════════════════════════════════════════════════════════════
print('=' * 60)
print(f'FINAL OOF AUC : {final_auc:.6f}')
print(f'V4 baseline   : 0.947622')
print(f'Delta         : {final_auc - 0.947622:+.6f}')
print('=' * 60)

submission = sample_sub.copy()
submission[TARGET] = final_test['stack'].values
submission.to_csv('submission.csv', index=False)

print('\nsubmission.csv saved ✓')
submission.head()

In [ ]:
# ─── Leaderboard comparison plot ──────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

versions = ['V1\nXGB blend', 'V2\nET+XGB', 'V3\nSmoke stack', 'V4\nLean stack', 'V5\n(this run)']
scores   = [0.940, 0.944, 0.945, 0.9476, final_auc]
colors   = ['#4a9eda','#4a9eda','#4a9eda','#f0a500', '#2ecc71']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(versions, scores, color=colors, width=0.55, zorder=3)
ax.set_ylim(0.935, max(scores) + 0.005)
ax.axhline(0.9476, color='#f0a500', linestyle='--', linewidth=1.5, label='V4 baseline')
ax.set_ylabel('OOF ROC-AUC', fontsize=12)
ax.set_title('F1 Pitstop — Model Progression', fontsize=14, fontweight='bold')
ax.yaxis.grid(True, alpha=0.3, zorder=0)
ax.set_axisbelow(True)
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0003,
            f'{score:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('v5_progress.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved ✓')